In [1]:
import subprocess, sys

In [2]:
subprocess.run(["nvidia-smi"])

Mon Jun 15 18:03:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [3]:
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "speechbrain",
    "accelerate"
])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 45.2 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: ruamel.yaml
    Found existing installation: ruamel.yaml 0.19.1
    Uninstalling ruamel.yaml-0.19.1:
      Successfully uninstalled ruamel.yaml-0.19.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'speechbrain', 'accelerate'], returncode=0)

In [4]:
import os
import torch
import datasets
from typing import Any
from functools import partial
from IPython.display import Audio
from dataclasses import dataclass
from transformers import pipeline
from transformers import Seq2SeqTrainer
from transformers import SpeechT5HifiGan
from transformers import SpeechT5Processor
from transformers import SpeechT5ForTextToSpeech
from transformers import Seq2SeqTrainingArguments
from speechbrain.pretrained import EncoderClassifier

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface_api")
login(token=hf_token)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

/tmp/ipykernel_23/191384018.py:14: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderClassifier


In [5]:
@dataclass
class Config:
    sampling_rate:int
    model_checkpoint:str
    speech_model:str

config = Config(sampling_rate=16_000, 
                model_checkpoint="microsoft/speecht5_tts",
                speech_model="speechbrain/spkrec-xvect-voxceleb")

config

Config(sampling_rate=16000, model_checkpoint='microsoft/speecht5_tts', speech_model='speechbrain/spkrec-xvect-voxceleb')

In [6]:
data = datasets.load_dataset("qmeeus/voxpopuli", "nl", split="train")

data

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

nl/train-00000-of-00021-6c22031eedbb80c4(…):   0%|          | 0.00/416M [00:00<?, ?B/s]

nl/train-00001-of-00021-cad841a50344627a(…):   0%|          | 0.00/415M [00:00<?, ?B/s]

...

nl/validation-00001-of-00002-6568c0fe2e6(…):   0%|          | 0.00/251M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20968 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1230 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

Dataset({
    features: ['audio_id', 'audio', 'text', 'language'],
    num_rows: 20968
})

In [7]:
data_casted = data.cast_column("audio", datasets.Audio(sampling_rate=config.sampling_rate))

data_casted

Dataset({
    features: ['audio_id', 'audio', 'text', 'language'],
    num_rows: 20968
})

In [8]:
processor = SpeechT5Processor.from_pretrained(config.model_checkpoint)

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

In [9]:
def extract_all_chars(batch):
    all_text = " ".join(batch["text"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}


vocabs = data_casted.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=data_casted.column_names,
)

dataset_vocab = set(vocabs["vocab"][0])
tokenizer_vocab = {k for k, _ in processor.tokenizer.get_vocab().items()}

Map:   0%|          | 0/20968 [00:00<?, ? examples/s]

In [10]:
print(dataset_vocab - tokenizer_vocab)

{'ș', 'Ò', 'Ü', 'Ö', 'ë', 'Š', '3', 'ņ', 'ğ', 'á', '´', '4', 'ß', '1', 'ì', 'à', 'ä', 'ó', '°', 'ü', 'ö', 'ô', 'È', 'ό', 'č', '7', 'ắ', 'š', '8', 'í', 'ć', 'Ó', 'ç', 'è', '6', 'ş', '0', '2', 'ž', 'β', 'Ž', '9', 'É', 'ï', 'ú', ' ', 'ñ', 'ę', '5', 'ò'}


In [11]:
replacements = [
    ("à", "a"),
    ("ç", "c"),
    ("è", "e"),
    ("ë", "e"),
    ("í", "i"),
    ("ï", "i"),
    ("ö", "o"),
    ("ü", "u"),
]


def cleanup_text(inputs):
    for src, dst in replacements:
        inputs["text"] = inputs["text"].replace(src, dst)
    return inputs


dataset = data_casted.map(cleanup_text)

Map:   0%|          | 0/20968 [00:00<?, ? examples/s]

In [12]:
from collections import defaultdict

speaker_counts = defaultdict(int)

for speaker_id in dataset["audio_id"]:
    speaker_counts[speaker_id] += 1

In [13]:
def select_speaker(speaker_id):
    return 100 <= speaker_counts[speaker_id] <= 400


data_casted_filt = data_casted.filter(select_speaker, input_columns=["audio_id"])

Filter:   0%|          | 0/20968 [00:00<?, ? examples/s]

In [14]:
spk_model_name = config.speech_model

device = "cuda" if torch.cuda.is_available() else "cpu"
speaker_model = EncoderClassifier.from_hparams(
    source=spk_model_name,
    run_opts={"device": device},
    savedir=os.path.join("/tmp", spk_model_name),
)


def create_speaker_embedding(waveform):
    with torch.no_grad():
        speaker_embeddings = speaker_model.encode_batch(torch.tensor(waveform))
        speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2)
        speaker_embeddings = speaker_embeddings.squeeze().cpu().numpy()
    return speaker_embeddings

hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/16.9M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/3.20k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/15.9M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


In [15]:
def prepare_dataset(example):
    audio = example["audio"]

    example = processor(
        text=example["text"],
        audio_target=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_attention_mask=False,
    )

    # strip off the batch dimension
    example["labels"] = example["labels"][0]

    # use SpeechBrain to obtain x-vector
    example["speaker_embeddings"] = create_speaker_embedding(audio["array"])

    return example

In [16]:
processed_example = prepare_dataset(dataset[0])
list(processed_example.keys())

['input_ids', 'labels', 'speaker_embeddings']

In [17]:
dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

Map:   0%|          | 0/20968 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (728 > 600). Running this sequence through the model will result in indexing errors


In [18]:
def is_not_too_long(input_ids):
    input_length = len(input_ids)
    return input_length < 200


dataset = dataset.filter(is_not_too_long, input_columns=["input_ids"])
len(dataset)

Filter:   0%|          | 0/20968 [00:00<?, ? examples/s]

17624

In [19]:
dataset = dataset.train_test_split(test_size=0.1)

In [20]:
@dataclass
class TTSDataCollatorWithPadding:
    processor: Any

    def __call__(
        self, features
    ):
        input_ids = [{"input_ids": feature["input_ids"]} for feature in features]
        label_features = [{"input_values": feature["labels"]} for feature in features]
        speaker_features = [feature["speaker_embeddings"] for feature in features]

        # collate the inputs and targets into a batch
        batch = processor.pad(
            input_ids=input_ids, labels=label_features, return_tensors="pt"
        )

        # replace padding with -100 to ignore loss correctly
        batch["labels"] = batch["labels"].masked_fill(
            batch.decoder_attention_mask.unsqueeze(-1).ne(1), -100
        )

        # not used during fine-tuning
        del batch["decoder_attention_mask"]

        # round down target lengths to multiple of reduction factor
        if model.config.reduction_factor > 1:
            target_lengths = torch.tensor(
                [len(feature["input_values"]) for feature in label_features]
            )
            target_lengths = target_lengths.new(
                [
                    length - length % model.config.reduction_factor
                    for length in target_lengths
                ]
            )
            max_length = max(target_lengths)
            batch["labels"] = batch["labels"][:, :max_length]

        # also add in the speaker embeddings
        batch["speaker_embeddings"] = torch.tensor(speaker_features)

        return batch

In [21]:
data_collator = TTSDataCollatorWithPadding(processor=processor)

In [22]:
model = SpeechT5ForTextToSpeech.from_pretrained(config.model_checkpoint)

pytorch_model.bin:   0%|          | 0.00/585M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# disable cache during training since it's incompatible with gradient checkpointing
model.config.use_cache = False

# set language and task for generation and re-enable cache
model.generate = partial(model.generate, use_cache=True)

In [24]:
training_args = Seq2SeqTrainingArguments(
    output_dir="speecht5_finetuned_voxpopuli_nl",  
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    num_train_epochs=2,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=2,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    # report_to=["tensorboard"],
    load_best_model_at_end=True,
    greater_is_better=False,
    label_names=["labels"],
    push_to_hub=False,
)

In [25]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=processor,
)

In [26]:
trainer.train()

Step,Training Loss,Validation Loss
1000,4.139802,0.478507
2000,3.952451,0.462725
3000,3.908049,0.459352
4000,3.889671,0.457903


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4000, training_loss=4.169603214263916, metrics={'train_runtime': 14177.2716, 'train_samples_per_second': 18.057, 'train_steps_per_second': 0.282, 'total_flos': 3.768140870983685e+16, 'train_loss': 4.169603214263916, 'epoch': 16.129097327281897})

In [27]:
tts = pipeline(
    "text-to-speech",
    model=trainer.model,
    tokenizer=processor.tokenizer,          
    feature_extractor=processor.feature_extractor,
)

config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/50.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

In [28]:
text = "hallo allemaal, ik praat nederlands. groetjes aan iedereen!"

In [29]:
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to("cuda:0")

tts = pipeline(
    "text-to-speech",
    model=trainer.model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    vocoder=vocoder,        
)

example = dataset["test"][304]
speaker_embeddings = torch.tensor(example["speaker_embeddings"], device="cuda:0").unsqueeze(0)

output = tts(text, forward_params={"speaker_embeddings": speaker_embeddings})

model.safetensors:   0%|          | 0.00/50.6M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

In [30]:
from IPython.display import Audio

Audio(output["audio"], rate=output["sampling_rate"])